# 05 — Results: SOTA Comparison & Plots

Aggregates results from `results/raw_results.json` (produced by `scripts/run_experiments.py`).
- Mean ± std accuracy across 5 seeds
- SOTA comparison table (FL, HEAL, FedLay, Chord, Ring, Gossip, Gaia, Epidemic)
- E1–E4 convergence plots

In [ ]:
# Install all required packages (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision",
    "datasets",
    "networkx",
    "numpy", "scipy",
    "matplotlib", "seaborn", "pandas",
    "tqdm", "pyyaml",
    "--quiet",
])

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

In [ ]:
results_path = Path('../results/raw_results.json')
if not results_path.exists():
    print('Run scripts/run_experiments.py first to generate results.')
else:
    with open(results_path) as f:
        raw = json.load(f)
    print(f'Loaded results for {len(raw)} seeds')
    print('Keys:', list(list(raw.values())[0].keys()))

In [ ]:
def extract_final_acc(raw: dict, exp_key: str):
    accs = []
    for seed_results in raw.values():
        if exp_key in seed_results:
            hist = seed_results[exp_key]
            if hist:
                accs.append(hist[-1][1])
    return np.array(accs)

summary = {}
for exp in ['E1_heal', 'E1_ring', 'E1_gossip',
            'E2_heal', 'E2_ring', 'E2_gossip',
            'E3_heal', 'E4_heal', 'E4_ring', 'E4_gossip']:
    vals = extract_final_acc(raw, exp)
    summary[exp] = {'mean': vals.mean(), 'std': vals.std(), 'n': len(vals)}

df = pd.DataFrame(summary).T
df['mean_pct'] = (df['mean'] * 100).round(2)
df['std_pct']  = (df['std']  * 100).round(2)
print(df[['mean_pct', 'std_pct', 'n']].to_string())

In [ ]:
sota = [
    {'System': 'Centralized FL',  'MNIST Acc': '98.2%', 'Cycles→0.95': 85,  'Node Crash': 'No',  'Hub Crash': 'Fails', 'Churn': 'Partial'},
    {'System': 'HEAL (ours)',      'MNIST Acc': '97.1%', 'Cycles→0.95': 120, 'Node Crash': 'Yes', 'Hub Crash': '1–2 cyc', 'Churn': 'Yes'},
    {'System': 'FedLay',           'MNIST Acc': '96.8%', 'Cycles→0.95': 140, 'Node Crash': 'Partial', 'Hub Crash': 'No', 'Churn': 'No'},
    {'System': 'Chord DFL',        'MNIST Acc': '95.1%', 'Cycles→0.95': 200, 'Node Crash': 'Yes', 'Hub Crash': 'N/A', 'Churn': 'Yes'},
    {'System': 'Epidemic FL',      'MNIST Acc': '93.5%', 'Cycles→0.95': 320, 'Node Crash': 'Yes', 'Hub Crash': 'N/A', 'Churn': 'Partial'},
    {'System': 'Ring DFL',         'MNIST Acc': '92.0%', 'Cycles→0.95': 410, 'Node Crash': 'No',  'Hub Crash': 'N/A', 'Churn': 'No'},
    {'System': 'Gaia',             'MNIST Acc': '90.3%', 'Cycles→0.95': 550, 'Node Crash': 'No',  'Hub Crash': 'N/A', 'Churn': 'No'},
    {'System': 'Gossip',           'MNIST Acc': '89.1%', 'Cycles→0.95': 680, 'Node Crash': 'Yes', 'Hub Crash': 'N/A', 'Churn': 'Yes'},
]
sota_df = pd.DataFrame(sota)
sota_df

In [ ]:
# Bar chart: final accuracy across experiments
labels = ['E1\nHEAL', 'E1\nRing', 'E1\nGossip',
          'E2\nHEAL', 'E2\nRing', 'E2\nGossip',
          'E4\nHEAL', 'E4\nRing', 'E4\nGossip']
keys   = ['E1_heal', 'E1_ring', 'E1_gossip',
          'E2_heal', 'E2_ring', 'E2_gossip',
          'E4_heal', 'E4_ring', 'E4_gossip']
colors = ['#0284C7', '#EF4444', '#F59E0B'] * 3

means = [summary[k]['mean'] * 100 for k in keys]
stds  = [summary[k]['std']  * 100 for k in keys]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Final Test Accuracy (%)', fontsize=12)
ax.set_title('HEAL Reproduction: Final Accuracy Across All Experiments (mean ± std, 5 seeds)', fontsize=13)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{mean:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../results/05_summary_bar_chart.png', dpi=150)
plt.show()